# DualTSR E-MMDiT Colab Training

This notebook is the Colab entry point for the E-MMDiT-based DualTSR training project. It installs the repo, downloads required assets, prepares a runtime config, and trains with resumable checkpoints saved to Google Drive.

## 1. Runtime

In Colab, select `Runtime > Change runtime type > GPU` before running the notebook.

In [ ]:
import os
import subprocess
from pathlib import Path

def run(cmd, cwd=None):
    print('+', ' '.join(str(x) for x in cmd))
    subprocess.run([str(x) for x in cmd], cwd=cwd, check=True)

run(['nvidia-smi'])

## 2. Clone and Install

In [ ]:
REPO_URL = 'https://github.com/yqwu905/DualTSR_Repro.git'
BRANCH = 'codex/emmdit'
REPO_DIR = Path('/content/DualTSR_Repro')

if not REPO_DIR.exists():
    run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)])
else:
    run(['git', 'fetch', 'origin'], cwd=REPO_DIR)
    run(['git', 'checkout', BRANCH], cwd=REPO_DIR)
    run(['git', 'pull', '--ff-only'], cwd=REPO_DIR)

os.chdir(REPO_DIR)
run(['python3', '-m', 'pip', 'install', '-U', 'pip'])
run(['python3', '-m', 'pip', 'install', '-r', 'requirements.txt'])

## 3. Configure Run

`DATASET='synth'` is fully automatic and uses open-licensed fonts plus online rendering. `DATASET='ctr'` expects `CTR_URL` or explicit LMDB paths.

In [ ]:
USE_DRIVE = True
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/DualTSR_Repro'

DATASET = 'synth'  # 'synth' or 'ctr'
CTR_URL = ''       # Google Drive folder/file URL or HTTP URL for CTR data, used only when DATASET='ctr'
CTR_TRAIN_LMDB = ''
CTR_VAL_LMDB = ''

RUN_NAME = 'colab_emmdit_synth'
MAX_STEPS = 10000
SAVE_EVERY = 500
BATCH_SIZE = 1
GLOBAL_BATCH_SIZE = 8
NUM_WORKERS = 2
PRECISION = 'bf16'

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    Path(DRIVE_PROJECT_DIR).mkdir(parents=True, exist_ok=True)
else:
    DRIVE_PROJECT_DIR = ''

## 4. Download Assets and Build Runtime Config

This downloads open fonts, prefetches the Nitro-E DC-AE base weights, prepares vocab/data, and writes `configs/train/colab_runtime.yaml`.

In [ ]:
prepare_cmd = [
    'python3', 'scripts/colab_prepare.py',
    '--base-config', 'configs/train/colab_emmdit_synth.yaml',
    '--out-config', 'configs/train/colab_runtime.yaml',
    '--dataset', DATASET,
    '--run-name', RUN_NAME,
    '--max-steps', MAX_STEPS,
    '--save-every', SAVE_EVERY,
    '--batch-size', BATCH_SIZE,
    '--global-batch-size', GLOBAL_BATCH_SIZE,
    '--num-workers', NUM_WORKERS,
    '--precision', PRECISION,
    '--device', 'cuda',
]
if DRIVE_PROJECT_DIR:
    prepare_cmd += ['--drive-root', DRIVE_PROJECT_DIR]
if DATASET == 'ctr':
    if CTR_URL:
        prepare_cmd += ['--ctr-url', CTR_URL]
    if CTR_TRAIN_LMDB:
        prepare_cmd += ['--ctr-train-lmdb', CTR_TRAIN_LMDB]
    if CTR_VAL_LMDB:
        prepare_cmd += ['--ctr-val-lmdb', CTR_VAL_LMDB]

run(prepare_cmd, cwd=REPO_DIR)

## 5. Train or Resume

`--resume auto` resumes from `<output_dir>/checkpoints/latest.pt` when it exists. With Drive enabled, checkpoints survive Colab runtime resets.

In [ ]:
run(['python3', 'train.py', '--config', 'configs/train/colab_runtime.yaml', '--resume', 'auto'], cwd=REPO_DIR)

## 6. Monitor

In [ ]:
import yaml

with open(REPO_DIR / 'configs/train/colab_runtime.yaml', 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)
log_dir = Path(cfg['output_dir']) / 'tensorboard'
print('TensorBoard log dir:', log_dir)
%load_ext tensorboard
%tensorboard --logdir $log_dir

## 7. Checkpoint Files

In [ ]:
with open(REPO_DIR / 'configs/train/colab_runtime.yaml', 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)
ckpt_dir = Path(cfg['output_dir']) / 'checkpoints'
print('Latest checkpoint:', ckpt_dir / 'latest.pt')
for path in sorted(ckpt_dir.glob('*.pt'))[-10:]:
    print(path)